In [ ]:
# Run this cell only one time: installing packages
%pip install selenium
%pip install Webdriver_manager
%pip install pandas
%pip install tqdm
%pip install Pyarrow
%pip install ipywidgets

In [10]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import Select
from selenium.webdriver.edge.options import Options
import pandas as pd
import time
from tqdm.notebook import tqdm
import re
import tempfile

t = 0.3
tt = 1.5

In [11]:
# launch the web browser
driver = webdriver.Chrome()
driver.get("https://eservices.drives.ga.gov/_/")
time.sleep(5)
driver.find_element(By.XPATH, '//span[text()="View County Registration Statistics"]').click()

In [12]:
# create an empty data frame
df = pd.DataFrame(columns=['county', 'total_vehicle', 'ev', 'passenger_vehicle',
                           'truck', 'trailer', 'motorcycle', 'bus', 'other'])

In [13]:
select_element = driver.find_element(By.XPATH, '//*[@id="Dd-6"]')
select_object = Select(select_element)
len(select_object.options)

160

In [ ]:
# replace with the year you want to scrape
year = 2023

year_box = driver.find_element(By.XPATH, '//*[@id="Dd-7"]')

current_year = year_box.get_attribute("value")

if str(year) != current_year:

    year_box.clear()
    year_box.send_keys(str(year))

    driver.find_element(By.XPATH, '//*[@id="Dd-6"]').click()

    time.sleep(tt)

In [14]:
select_element = driver.find_element(By.XPATH, '//*[@id="Dd-6"]')
select_object = Select(select_element)

# collect the data by county through the loop
for i in tqdm(range(len(select_object.options))):

    # choose a county
    select_element = driver.find_element(By.XPATH, '//*[@id="Dd-6"]')
    select_object = Select(select_element)
    time.sleep(t)

    county = select_object.options[i].text
    time.sleep(t)
    select_object.options[i].click()
    time.sleep(t)
    

    driver.find_element(By.XPATH, '//*[@id="Dd-8"]').click()
    time.sleep(tt)

    # get texts
    t1 = driver.find_element(By.CSS_SELECTOR, '#caption2_Dd-5').text
    time.sleep(t)
    t2 = driver.find_element(By.CSS_SELECTOR, '#caption2_Dd-a').text
    time.sleep(t)
    t3 = driver.find_element(By.CSS_SELECTOR, '#caption2_Dd-b').text
    time.sleep(t)
    t4 = driver.find_element(By.CSS_SELECTOR, '#caption2_Dd-c').text
    time.sleep(t)
    t5 = driver.find_element(By.CSS_SELECTOR, '#caption2_Dd-d').text
    time.sleep(t)
    t6 = driver.find_element(By.CSS_SELECTOR, '#caption2_Dd-e').text
    time.sleep(t)
    t7 = driver.find_element(By.CSS_SELECTOR, '#caption2_Dd-f').text
    time.sleep(t)

    # extract and save numbers
    row_to_add = pd.DataFrame(
        [{'county': county,
        'total_vehicle': re.findall(r'\d+',t1)[2],
        'ev': re.findall(r'\d+',t1)[3],
        'passenger_vehicle': re.findall(r'\d+',t2)[0],
        'truck': re.findall(r'\d+',t3)[0],
        'trailer': re.findall(r'\d+',t4)[0],
        'motorcycle': re.findall(r'\d+',t5)[0],
        'bus': re.findall(r'\d+',t6)[0],
        'other': re.findall(r'\d+',t7)[0]}])

    # append
    df = pd.concat([df, row_to_add], ignore_index=True)
    time.sleep(t)

  0%|          | 0/160 [00:00<?, ?it/s]

In [ ]:
# save the result
df.to_csv(f'registered_vehicles_by_county_{year}.csv')